In [1]:
import os
# 모든 단계는 저장소 루트의 data/ 아래를 공유한다 (노트북은 cwd가 자신의 폴더라고 가정).
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
CORPUS_DATA = os.path.join(REPO_ROOT, "data", "1.corpus")

import os, re, json, html, random, string, gc
from collections import Counter
from pathlib import Path
from tqdm import tqdm

# ==========================================
# [설정]
# ==========================================
BASE_DIR = os.path.join(CORPUS_DATA, "2.aihub/origin")

# 국문 원천데이터 JSON 디렉토리
JSON_DIRS = [
    f'{BASE_DIR}/08.전문 의학지식 데이터/Training/01.원천데이터/TS_국문_기타',
    f'{BASE_DIR}/08.전문 의학지식 데이터/Training/01.원천데이터/TS_국문_온라인 의료 정보 제공 사이트',
    f'{BASE_DIR}/08.전문 의학지식 데이터/Training/01.원천데이터/TS_국문_의학 교과서',
    f'{BASE_DIR}/08.전문 의학지식 데이터/Training/01.원천데이터/TS_국문_학술 논문 및 저널',
    f'{BASE_DIR}/08.전문 의학지식 데이터/Training/01.원천데이터/TS_국문_학회 가이드라인',
    f'{BASE_DIR}/09.필수의료 의학지식 데이터/3.개방데이터/1.데이터/Training/01.원천데이터/TS_국문_기타',
    f'{BASE_DIR}/09.필수의료 의학지식 데이터/3.개방데이터/1.데이터/Training/01.원천데이터/TS_국문_온라인 의료 정보 제공 사이트',
    f'{BASE_DIR}/09.필수의료 의학지식 데이터/3.개방데이터/1.데이터/Training/01.원천데이터/TS_국문_의학 교과서',
    f'{BASE_DIR}/09.필수의료 의학지식 데이터/3.개방데이터/1.데이터/Training/01.원천데이터/TS_국문_학술 논문 및 저널',
    f'{BASE_DIR}/09.필수의료 의학지식 데이터/3.개방데이터/1.데이터/Training/01.원천데이터/TS_국문_학회 가이드라인',
]

# 154 의료/법률 서적 TXT
TXT_DIRS = [
    f'{BASE_DIR}/154.의료, 법률 전문 서적 말뭉치/Training/01.원천데이터/TS_01.의료',
    f'{BASE_DIR}/154.의료, 법률 전문 서적 말뭉치/Validation/01.원천데이터/VS_01.의료'
]

# 출력 파일 설정
OUTPUT_DIR = os.path.join(CORPUS_DATA, "2.aihub/outputs")
MERGED_OUTPUT_FILE = f'{OUTPUT_DIR}/merged_book.txt'

SEED = 42
random.seed(SEED)

os.makedirs(OUTPUT_DIR, exist_ok=True)
print('설정 완료')
print('=' * 60)

json_files, txt_files = [], []
for d in JSON_DIRS:
    p = Path(d)
    if p.exists():
        fs = list(p.glob('*.json'))
        json_files.extend(fs)
        print(f'  JSON [{len(fs):>6}]  {p.name}')
    else:
        print(f'  [MISSING] {d}')

for d in TXT_DIRS:
    p = Path(d)
    if p.exists():
        fs = list(p.glob('*.txt'))
        txt_files.extend(fs)
        print(f'  TXT  [{len(fs):>6}]  {p.name}')
    else:
        print(f'  [MISSING] {d}')
print('-' * 60)
print(f'  총 JSON: {len(json_files):,}  |  총 TXT: {len(txt_files):,}  |  합계: {len(json_files)+len(txt_files):,}')

# ==========================================
# [단계 1] 텍스트 정제 함수 정의
# ==========================================
PUNCT_MAPPING = {
    '\u2018': "'", '\u2019': "'", '\u201c': '"', '\u201d': '"',
    '\u20b9': 'e', '\u00b4': "'", '\u00b0': '', '\u20ac': 'e', '\u2122': 'tm',
    '\u221a': ' sqrt ', '\u00d7': 'x', '\u00b2': '2', '\u2014': '-', '\u2013': '-',
    '\u00a3': 'e', '\u221e': 'infinity', '\u03b8': 'theta', '\u00f7': '/',
    '\u03b1': 'alpha', '\u2022': '.', '\u00e0': 'a', '\u2212': '-',
    '\u03b2': 'beta', '\u2205': '', '\u00b3': '3', '\u03c0': 'pi',
    '\u200b': ' ', '\u2026': ' ... ', '\ufeff': '', '\u00b7': '.'
}
_PUNCT     = re.escape(string.punctuation)
_WHITELIST = re.compile(rf'[^A-Za-z0-9\u3131-\u3163\uac00-\ud7a3{_PUNCT}\s]')
_HTML      = re.compile(r'<[^>]*>')
_SPACE     = re.compile(r'\s+')

def clean_str(text: str) -> str:
    if not isinstance(text, str):
        return ''
    text = _HTML.sub('', text)
    for p, r in PUNCT_MAPPING.items():
        text = text.replace(p, r)
    text = _WHITELIST.sub('', text)
    return _SPACE.sub(' ', text).strip()

def clean_text_medical(text: str) -> str:
    if not isinstance(text, str) or not text:
        return ''
    text = html.unescape(text)
    text = re.sub(r'(https?://|www\.)[A-Za-z0-9\uac00-\ud7a3./?=&%~_#-]+', '', text)
    text = re.sub(r'[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}', '', text)
    text = re.sub(r' ?\(\s*\) ?', ' ', text)
    text = re.sub(r'\(\.+\)', '', text)
    text = re.sub(r'\.{2,}', '.', text)
    text = re.sub(r'외부\s*링크[:.]*', '', text)
    text = re.sub(r'Copyright.*', '', text, flags=re.IGNORECASE)
    return clean_str(text)

def is_spam(text: str) -> bool:
    length = len(text)
    if length < 20:
        return True
    boilerplate = ['저작권', '크리에이티브 커먼즈', '라이선스', '분류:', '목차',
                   '파일:', '각주', '편집', '토론', '역사']
    if length < 50 and any(k in text for k in boilerplate):
        return True
    if text.strip().startswith(('*', '>', '-')) and len(text.strip()[1:].strip()) < 5:
        return True
    if length < 300:
        return False
    ns = text.replace(' ', '')
    if not ns:
        return True
    valid = len(re.sub(r'[^\uac00-\ud7a3a-zA-Z]', '', ns))
    if valid / len(ns) < 0.20:
        return True
    words = text.split()
    if words:
        _, cnt = Counter(words).most_common(1)[0]
        if cnt / len(words) > 0.6:
            return True
    if length > 1000 and text.count(' ') / length < 0.01:
        return True
    return False

def sentence_split(text: str):
    sents = re.split(r'(?<=[.?!])\s+', text)
    return [s.strip() for s in sents if len(s.strip()) > 5]

# ==========================================
# [단계 2] 텍스트 추출 및 병합 (merged_book.txt)
# ==========================================
print(f'\n[병합 시작] 텍스트를 정제하여 {MERGED_OUTPUT_FILE} 에 저장합니다...')

# 파일 쓰기 모드('w')로 열어서 메모리 부하 없이 실시간으로 기록
with open(MERGED_OUTPUT_FILE, 'w', encoding='utf-8') as out_f:
    
    # 1. JSON 파일 처리
    for fp in tqdm(json_files, desc="JSON 파일 병합 중"):
        try:
            with open(fp, encoding='utf-8-sig') as f:
                d = json.load(f)
            
            # JSON 포맷에 따라 'content', 'text' 등 적절한 키 사용
            content = d.get('content', '') 
            
            if content:
                cleaned = clean_text_medical(content)
                if not is_spam(cleaned):
                    sents = sentence_split(cleaned)
                    if sents:
                        # 문장 단위로 줄바꿈하여 기록하고, 문서 간 구분을 위해 빈 줄 추가
                        out_f.write('\n'.join(sents) + '\n\n')
        except Exception as e:
            # 깨진 JSON 파일 등은 무시하고 계속 진행
            continue

    # 2. TXT 파일 처리
    for fp in tqdm(txt_files, desc="TXT 파일 병합 중"):
        try:
            with open(fp, encoding='utf-8-sig') as f:
                content = f.read()
            
            if content:
                cleaned = clean_text_medical(content)
                if not is_spam(cleaned):
                    sents = sentence_split(cleaned)
                    if sents:
                        out_f.write('\n'.join(sents) + '\n\n')
        except Exception as e:
            continue

print(f'\n[병합 완료] 모든 텍스트가 {MERGED_OUTPUT_FILE} 에 저장되었습니다.')

설정 완료
  [MISSING] ./origin/08.전문 의학지식 데이터/Training/01.원천데이터/TS_국문_기타
  [MISSING] ./origin/08.전문 의학지식 데이터/Training/01.원천데이터/TS_국문_온라인 의료 정보 제공 사이트
  [MISSING] ./origin/08.전문 의학지식 데이터/Training/01.원천데이터/TS_국문_의학 교과서
  [MISSING] ./origin/08.전문 의학지식 데이터/Training/01.원천데이터/TS_국문_학술 논문 및 저널
  [MISSING] ./origin/08.전문 의학지식 데이터/Training/01.원천데이터/TS_국문_학회 가이드라인
  JSON [  1032]  TS_국문_기타
  JSON [  2034]  TS_국문_온라인 의료 정보 제공 사이트
  JSON [   256]  TS_국문_의학 교과서
  JSON [  9685]  TS_국문_학술 논문 및 저널
  JSON [  1157]  TS_국문_학회 가이드라인
  [MISSING] ./origin/154.의료, 법률 전문 서적 말뭉치/Training/01.원천데이터/TS_01.의료
  [MISSING] ./origin/154.의료, 법률 전문 서적 말뭉치/Validation/01.원천데이터/VS_01.의료
------------------------------------------------------------
  총 JSON: 14,164  |  총 TXT: 0  |  합계: 14,164

[병합 시작] 텍스트를 정제하여 ./outputs/merged_book.txt 에 저장합니다...


JSON 파일 병합 중: 100%|█████████████████████████████████████████████████████████| 14164/14164 [00:18<00:00, 754.11it/s]
TXT 파일 병합 중: 0it [00:00, ?it/s]


[병합 완료] 모든 텍스트가 ./outputs/merged_book.txt 에 저장되었습니다.
